In [1]:
!pip install -q -U transformers accelerate peft trl bitsandbytes

In [2]:
# 2. Imports
import torch, json, os
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [3]:
TRAIN_PATH = "/content/train_split.jsonl"
VAL_PATH = "/content/val_split.jsonl"

assert os.path.exists(TRAIN_PATH), f"Missing {TRAIN_PATH} - upload it or clone the repo first"


In [4]:
# 4. Model + tokenizer (4-bit quantized base for memory-efficient fine-tuning)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # small, open-source, instruction-tuned, Apache-2.0

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [6]:
raw = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})

def format_example(ex):
    messages = [
        {"role": "system", "content": "You are a meticulous data-cleaning assistant. "
                                       "Given a dataset summary, produce prioritized, "
                                       "actionable cleaning recommendations."},
        {"role": "user", "content": f"{ex['instruction']}\n\n{ex['input']}"},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = raw["train"].map(format_example, remove_columns=raw["train"].column_names)
val_ds = raw["validation"].map(format_example, remove_columns=raw["validation"].column_names)

print(train_ds[0]["text"][:800])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

<|im_start|>system
You are a meticulous data-cleaning assistant. Given a dataset summary, produce prioritized, actionable cleaning recommendations.<|im_end|>
<|im_start|>user
Review this dataset profile and suggest the cleaning steps needed before analysis or modeling.

Dataset: Taxi / travel trip logs
Rows: 15000, Columns: 9
Column names: trip_id, driver_id, pickup_time, drop_time, distance_km, fare_amount, vehicle_type, city, trip_rating

Observed data quality issues:
- 'vehicle_type' shows extreme outliers (max value 8x the 95th percentile)
- Target column 'driver_id' is highly imbalanced (95% vs 5%)
- 'trip_id' contains multiple date formats (e.g. 'DD/MM/YYYY' and 'YYYY-MM-DD' mixed)
- 'pickup_time' appears to mix units (some values ~1-10, others ~1000+, suggesting different units)
- '


In [7]:
import trl
print(trl.__version__)

1.9.0


In [8]:
sft_config = SFTConfig(
    output_dir="outputs/qwen2.5-0.5b-cleaning-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)


Adding EOS to train dataset:   0%|          | 0/234 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/234 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/234 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/234 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/234 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

In [9]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.020682,0.448860,0.705990,106934.000000,0.890368
2,0.177650,0.131245,0.209528,213868.000000,0.957560
3,0.113121,0.105793,0.163728,320802.000000,0.964841


TrainOutput(global_step=45, training_loss=0.6490783280796475, metrics={'train_runtime': 744.6805, 'train_samples_per_second': 0.943, 'train_steps_per_second': 0.06, 'total_flos': 849959532986880.0, 'train_loss': 0.6490783280796475, 'epoch': 3.0})

In [10]:
# 9. Save the LoRA adapter (small - a few MB) for use in inference/infer.py
ADAPTER_DIR = "outputs/qwen2.5-0.5b-cleaning-lora/final_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)

# Optional: zip and download, or push to a Hugging Face Hub repo / your GitHub repo (as a release asset,
# since LoRA adapters are small, or via git-lfs / HF Hub for larger ones).
!zip -r adapter.zip {ADAPTER_DIR}
from google.colab import files
files.download("adapter.zip")

Saved adapter to outputs/qwen2.5-0.5b-cleaning-lora/final_adapter
  adding: outputs/qwen2.5-0.5b-cleaning-lora/final_adapter/ (stored 0%)
  adding: outputs/qwen2.5-0.5b-cleaning-lora/final_adapter/README.md (deflated 65%)
  adding: outputs/qwen2.5-0.5b-cleaning-lora/final_adapter/adapter_model.safetensors (deflated 21%)
  adding: outputs/qwen2.5-0.5b-cleaning-lora/final_adapter/adapter_config.json (deflated 59%)
  adding: outputs/qwen2.5-0.5b-cleaning-lora/final_adapter/tokenizer_config.json (deflated 59%)
  adding: outputs/qwen2.5-0.5b-cleaning-lora/final_adapter/tokenizer.json (deflated 81%)
  adding: outputs/qwen2.5-0.5b-cleaning-lora/final_adapter/chat_template.jinja (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from peft import PeftModel

test_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
test_model = PeftModel.from_pretrained(test_model, ADAPTER_DIR)

sample_summary = '''Dataset: E-commerce orders
Rows: 8600, Columns: 11
Observed data quality issues:
- 'unit_price' has 12% missing values
- 'customer_id' shows extreme outliers (max value 15x the 95th percentile)
- 'shipping_city' has inconsistent category labels (e.g. 'NY', 'New York', 'new york' all mean the same thing)
'''

messages = [
    {"role": "system", "content": "You are a meticulous data-cleaning assistant. "
                                   "Given a dataset summary, produce prioritized, "
                                   "actionable cleaning recommendations."},
    {"role": "user", "content": f"Given the dataset summary below, list prioritized data-cleaning recommendations.\n\n{sample_summary}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(test_model.device)
out = test_model.generate(**inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Recommended cleaning steps:
1. Handle missing values in 'unit_price': if > 10%, consider missing or imputing with median/mode, then re-check for missingness before dropping, since 12% loss would bias the dataset.
2. Investigate outliers in 'customer_id': cap using IQR or winsorization at the 1st/99th percentile, or isolate them for separate analysis if they represent genuine rare events rather than data-entry errors (a 15x jump over the 95th percentile needs manual review, not automatic removal).
3. Normalize categories in 'shipping_city': lowercase/trim whitespace, then map known synonyms ('NY', 'New York', 'new york) to a single canonical label using a lookup table rather than fuzzy-matching blindly, to avoid merging genuinely distinct categories.
4. Version the raw and cleaned datasets separately so cleaning decisions can be revisited or rolled back.
